# Baseline Characterization — Elastocaloric Polymer Films

This notebook walks through the complete baseline analysis pipeline for a new polymer film candidate:

1. DSC — phase transition enthalpy ΔH and entropy change ΔS
2. Stress–strain — Young's modulus, maximum strain, hysteresis area
3. IR thermography — adiabatic temperature change ΔT_ad
4. Figure-of-merit comparison against reference materials

**Replace the file paths in Section 0 with your actual data before running.**

In [ ]:
import sys
sys.path.insert(0, '../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from dsc_analysis import analyze as dsc_analyze
from ir_thermography import load_stack, load_metadata, compute_delta_t, find_delta_t_ad, plot_delta_t_trace

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 0 — Configuration

In [ ]:
MATERIAL_NAME   = "Natural Rubber — 0.5 mm, 200% strain"
DSC_FILE        = Path('../../experiments/01_dsc/nr_200pct.csv')
TENSILE_FILE    = Path('../../experiments/02_tensile/nr_200pct.csv')
IR_FRAME_DIR    = Path('../../experiments/03_ir_thermography/nr_200pct/')
FILM_MASS_G     = 0.85   # weigh the film before mounting
FILM_AREA_CM2   = 10.0   # active area

## 1 — DSC: Phase Transition Enthalpy

In [ ]:
if DSC_FILE.exists():
    dsc_results = dsc_analyze(DSC_FILE, plot=True)
    print(f"\nDSC Results for {MATERIAL_NAME}:")
    for k, v in dsc_results.items():
        print(f"  {k:30s}: {v}")
else:
    print(f"DSC file not found: {DSC_FILE}\nPlace your DSC export CSV there and re-run.")
    dsc_results = {}

## 2 — Stress–Strain Analysis

In [ ]:
if TENSILE_FILE.exists():
    tens = pd.read_csv(TENSILE_FILE)
    strain = tens['strain_pct'].values
    stress = tens['stress_MPa'].values

    # Young's modulus from initial linear region (0–5 %)
    mask_lin = strain <= 5
    if mask_lin.sum() > 2:
        E_kPa = np.polyfit(strain[mask_lin] / 100, stress[mask_lin] * 1e3, 1)[0]
    else:
        E_kPa = np.nan

    # Hysteresis area (loading vs unloading)
    n_half = len(strain) // 2
    hysteresis_J_cm3 = abs(np.trapz(stress[:n_half], strain[:n_half] / 100)
                          - np.trapz(stress[n_half:], strain[n_half:] / 100))

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(strain, stress, lw=1.5, color='darkorange')
    ax.fill_between(strain[:n_half], stress[:n_half], stress[n_half:][::-1],
                    alpha=0.2, color='darkorange', label=f'Hysteresis = {hysteresis_J_cm3:.3f} MJ/m³')
    ax.set_xlabel('Strain (%)')
    ax.set_ylabel('Stress (MPa)')
    ax.set_title(f'Stress–Strain — {MATERIAL_NAME}')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"  Young's modulus (initial): {E_kPa:.1f} kPa")
    print(f"  Max stress:                {stress.max():.3f} MPa")
    print(f"  Max strain:                {strain.max():.1f} %")
    print(f"  Hysteresis area:           {hysteresis_J_cm3:.4f} MJ m⁻³")
else:
    print(f"Tensile file not found: {TENSILE_FILE}")

## 3 — IR Thermography: ΔT_ad

In [ ]:
if IR_FRAME_DIR.exists():
    stack = load_stack(IR_FRAME_DIR)
    meta  = load_metadata(IR_FRAME_DIR)
    delta_t = compute_delta_t(stack, ref_frame=0)
    ir_results = find_delta_t_ad(delta_t, meta)

    plot_delta_t_trace(delta_t, meta)

    print("\nIR Thermography Results:")
    for k, v in ir_results.items():
        print(f"  {k:30s}: {v}")
else:
    print(f"IR frame directory not found: {IR_FRAME_DIR}")
    ir_results = {}

## 4 — Figure of Merit Summary

In [ ]:
# Reference values from literature
reference_data = {
    'TiNiCuCo SMA film\n(Xu et al. 2024)': {'delta_T_ad_K': 20.0, 'cost_eur_m2': 500, 'max_strain_pct': 8},
    'NiTi SMA wire\n(Tušek et al. 2016)':  {'delta_T_ad_K': 15.0, 'cost_eur_m2': 200, 'max_strain_pct': 8},
    'Natural rubber\n(this work)':          {'delta_T_ad_K': ir_results.get('delta_T_ad_K', 0),
                                              'cost_eur_m2': 1, 'max_strain_pct': 400},
}

labels = list(reference_data.keys())
delta_ts = [v['delta_T_ad_K'] for v in reference_data.values()]
costs    = [v['cost_eur_m2']  for v in reference_data.values()]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

colors = ['steelblue', 'steelblue', 'firebrick']
axes[0].barh(labels, delta_ts, color=colors)
axes[0].set_xlabel('ΔT_ad (K)')
axes[0].set_title('Adiabatic temperature change')
axes[0].axvline(2, ls='--', color='gray', lw=0.8, label='Min. useful threshold')
axes[0].legend(fontsize=9)

axes[1].barh(labels, costs, color=colors)
axes[1].set_xlabel('Material cost (€ m⁻²)')
axes[1].set_title('Material cost comparison')
axes[1].set_xscale('log')

fig.suptitle(f'Figure of Merit — {MATERIAL_NAME}', fontsize=13)
fig.tight_layout()
plt.show()